# 02 · Retrieval / RAG index  _(owner: M2)_

Build the ChromaDB glossary index over MedlinePlus + lay-health guidelines; measure retrieval hit quality; wire it into the Simplifier.

In [2]:
import pandas as pd

df = pd.read_csv("../data/train.csv")

print(df.head())
print(df.columns)

   question      pmid                                         input_text  \
0         1  29857264  Exercise-Associated Muscle Cramps (EAMC) are a...   
1         1  33722257  Background: Muscle cramp is a painful, involun...   
2         1  31696455  Muscle cramp is a temporary but intense and pa...   
3         1  30168894  Muscular cramp is a common symptom in healthy ...   
4         1  29763070  Muscle cramps result in continuous, involuntar...   

                                         target_text  Adaptation_Version  \
0  Exercise-Associated Muscle Cramps (EAMC) are a...                   2   
1  Muscle cramps are unconscious contractions of ...                   2   
2  Muscle cramp is a temporary but intense, painf...                   2   
3  Muscle cramps are common in healthy people, es...                   2   
4  Muscle cramps cause constant and unintended co...                   2   

  Question_Type  
0             C  
1             C  
2             C  
3             

In [3]:
print("ORIGINAL MEDICAL TEXT:")
print(df.loc[0, "input_text"])

print("\nSIMPLIFIED TEXT:")
print(df.loc[0, "target_text"])

ORIGINAL MEDICAL TEXT:
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechanisms: the dehydration or electrolyte depletion mechanism, and the neuromuscular mechanism. The aim of this review is to examine the recent literature, in terms of physiological mechanisms of EAMC. A comprehensive search was conducted on PubMed and Google Scholar. The following terminology was applied: muscle cramps, neuromuscular hypothesis (or thesis), dehydration hypothesis, Exercise-Associated muscle cramps, nocturnal cramps, muscle spasm, muscle fatigue. From the initial literature of 424 manuscripts, sixty-nine manuscripts were included, analyzed, compared and summa

In [4]:
for i in range(3):
    print(f"\n=== PLABA EXAMPLE {i} ===")

    print("\nORIGINAL:")
    print(df.loc[i, "input_text"][:500])

    print("\nSIMPLIFIED:")
    print(df.loc[i, "target_text"][:500])


=== PLABA EXAMPLE 0 ===

ORIGINAL:
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechanisms: the dehydration or electrolyte depletion mechanism, and the neuromuscular mechanism. The aim o

SIMPLIFIED:
Exercise-Associated Muscle Cramps (EAMC) are a common type of muscle spasm, in which a muscle continually contracts without intention, causing pain. Scientists have tried to explain why these cramps happen, but have not been able to. An idea commonly returned to throughout the years is that EAMCs may be caused by heat. Recently, though, more likely explanations are thought to include dehydration, lack of electrolytes, or issues with the nerves co

In [5]:
# Get one example from each different question group
examples = df.drop_duplicates(subset="question")

# Look at the first five different questions
for _, row in examples.head(5).iterrows():

    print(f"\n=== QUESTION {row['question']} ===")

    print("\nORIGINAL:")
    print(row["input_text"][:400])

    print("\nSIMPLIFIED:")
    print(row["target_text"][:400])


=== QUESTION 1 ===

ORIGINAL:
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechan

SIMPLIFIED:
Exercise-Associated Muscle Cramps (EAMC) are a common type of muscle spasm, in which a muscle continually contracts without intention, causing pain. Scientists have tried to explain why these cramps happen, but have not been able to. An idea commonly returned to throughout the years is that EAMCs may be caused by heat. Recently, though, more likely explanations are thought to include dehydration, 

=== QUESTION 3 ===

ORIGINAL:
Hyperkalemia is a frequent clinical abnormality in patients with chronic kidney disease, and it is associated with higher r

In [6]:
medical_vocab = [
    {
        "term": "etiology",
        "aliases": ["cause of disease"],
        "definition": "The cause of a disease or medical condition."
    },
    {
        "term": "hyperkalemia",
        "aliases": ["high blood potassium"],
        "definition": "Higher-than-normal potassium in the blood."
    },
    {
        "term": "neuromuscular",
        "aliases": ["nerve and muscle"],
        "definition": "Related to nerves and muscles."
    }
]

for item in medical_vocab:
    print(item["term"], ":", item["definition"])

etiology : The cause of a disease or medical condition.
hyperkalemia : Higher-than-normal potassium in the blood.
neuromuscular : Related to nerves and muscles.


In [8]:
import chromadb
from pathlib import Path

# Find the project's data folder.
project_root = Path.cwd()

if not (project_root / "data").exists():
    project_root = project_root.parent

# Create a local database inside data/.
db_path = project_root / "data" / "chroma_db"

client = chromadb.PersistentClient(path=str(db_path))

# Create or open a vocabulary collection.
collection = client.get_or_create_collection(
    name="medical_vocabulary"
)

print("ChromaDB collection created!")


ChromaDB collection created!


In [ ]:
# Store the medical vocabulary in the collection.
collection.upsert(
    ids=[item["term"] for item in medical_vocab],
    documents=[
        item["term"] + ": " + item["definition"]
        for item in medical_vocab
    ],
    metadatas=[
        {"term": item["term"]}
        for item in medical_vocab
    ]
)

print("Stored entries:", collection.count())

/Users/andrewnguyen/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:33<00:00, 2.48MiB/s]


Stored entries: 3


In [ ]:
# Query the collection for a medical term
results = collection.query(
    query_texts=["high blood potassium"],
    n_results=1
)

print(results["documents"][0][0])

hyperkalemia: Higher-than-normal potassium in the blood.


In [ ]:
# Query the collection for multiple medical terms
queries = [
    "What causes a disease?",
    "Problems involving nerves and muscles",
    "Too much potassium in the bloodstream"
]

for query in queries:
    results = collection.query(
        query_texts=[query],
        n_results=1
    )

    print("Search:", query)
    print("Result:", results["documents"][0][0])
    print()

Search: What causes a disease?
Result: etiology: The cause of a disease or medical condition.

Search: Problems involving nerves and muscles
Result: neuromuscular: Related to nerves and muscles.

Search: Too much potassium in the bloodstream
Result: hyperkalemia: Higher-than-normal potassium in the blood.



In [12]:
additional_vocab = [
    {
        "term": "physiological mechanism",
        "definition": "How a process works inside the body."
    },
    {
        "term": "electrolyte depletion",
        "definition": "Losing too much of important minerals such as sodium or potassium."
    },
    {
        "term": "chronic kidney disease",
        "definition": "A long-term condition in which the kidneys are damaged or do not work as well as they should."
    },
    {
        "term": "malignant arrhythmias",
        "definition": "Dangerous abnormal heart rhythms that can be life-threatening."
    },
    {
        "term": "fasting plasma glucose",
        "definition": "A measurement of blood sugar after a period without eating."
    },
    {
        "term": "diagnostic criteria",
        "definition": "Medical rules or test results used to identify a condition."
    },
    {
        "term": "double-blind study",
        "definition": "A study where participants and certain researchers do not know which treatment each participant receives."
    }
]

# Combine the original and new vocabulary.
all_vocab = medical_vocab + additional_vocab

print("Total terms:", len(all_vocab))

Total terms: 10


In [18]:
collection.upsert(
    ids=[item["term"] for item in all_vocab],
    documents=[
        item["term"] + ": " + item["definition"]
        for item in all_vocab
    ]
)

print("Stored entries:", collection.count())

Stored entries: 10


In [19]:
results = collection.query(
    query_texts=["dangerous irregular heartbeat"],
    n_results = 3
)

for document in results["documents"][0]:
    print(document)

malignant arrhythmias: Dangerous abnormal heart rhythms that can be life-threatening.
etiology: The cause of a disease or medical condition.
hyperkalemia: Higher-than-normal potassium in the blood.


In [20]:
results = collection.query(
    query_texts=["dangerous irregular heartbeat"],
    n_results=3,
    include=["documents", "distances"]
)

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print(f"Distance: {distance:.4f} | {document}")

Distance: 0.6661 | malignant arrhythmias: Dangerous abnormal heart rhythms that can be life-threatening.
Distance: 1.4204 | etiology: The cause of a disease or medical condition.
Distance: 1.4684 | hyperkalemia: Higher-than-normal potassium in the blood.


In [ ]:
# Embeddings:
# Convert text into numerical vectors so computers can compare meanings + identify similar words or phrases.

# ChromaDB:
# A vector database that stores text + embeddings, allowing us to search for medical definitions by meaning using it's embedding model.

# Queries:
# Search requests sent to ChromaDB to find relevant information.
# Searching "high blood potassium" can retrieve the medical term "hyperkalemia".

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Use the first original medical passage from PLABA.
medical_text = df.iloc[0]["input_text"]

# Create a text splitter.
splitter = RecursiveCharacterTextSplitter(
    # 500 characters per chunk
    chunk_size=500,
    # 75 characters of overlap between chunks
    chunk_overlap=75
)

# Split the medical passage into smaller pieces.
chunks = splitter.split_text(medical_text)

print("Original length:", len(medical_text))
print("Number of chunks:", len(chunks))

# Display each chunk.
for i, chunk in enumerate(chunks):
    print(f"\n=== CHUNK {i + 1} ===")
    print(chunk)

Original length: 1558
Number of chunks: 4

=== CHUNK 1 ===
Exercise-Associated Muscle Cramps (EAMC) are a common painful condition of muscle spasms. Despite scientists tried to understand the physiological mechanism that underlies these common phenomena, the etiology is still unclear. From 1900 to nowadays, the scientific world retracted several times the original hypothesis of heat cramps. However, recent literature seems to focus on two potential mechanisms: the dehydration or electrolyte depletion mechanism, and the neuromuscular mechanism. The aim

=== CHUNK 2 ===
electrolyte depletion mechanism, and the neuromuscular mechanism. The aim of this review is to examine the recent literature, in terms of physiological mechanisms of EAMC. A comprehensive search was conducted on PubMed and Google Scholar. The following terminology was applied: muscle cramps, neuromuscular hypothesis (or thesis), dehydration hypothesis, Exercise-Associated muscle cramps, nocturnal cramps, muscle spasm, mus

In [ ]:
# Load and inspect the PLABA dataset from a JSON file.
import json
from pathlib import Path

path = Path("data/data.json")

if not path.exists():
    path = Path("../data/data.json")

with open(path, "r", encoding="utf-8") as file:
    plaba = json.load(file)

print("Data type:", type(plaba))
print("Number of top-level entries:", len(plaba))

if isinstance(plaba, dict):
    print("First 5 keys:", list(plaba.keys())[:5])
    first_entry = next(iter(plaba.values()))
else:
    first_entry = plaba[0]

print("First entry type:", type(first_entry))

if isinstance(first_entry, dict):
    print("First entry keys:", list(first_entry.keys()))

Data type: <class 'dict'>
Number of top-level entries: 75
First 5 keys: ['1', '2', '3', '4', '5']
First entry type: <class 'dict'>
First entry keys: ['15902691', '25432724', '29763070', '29857264', '30168894', '30237473', '31696455', '32956536', '33722257', '34346706', 'question', 'question_type']


In [ ]:
# Access the first question group and actual article within it
question_1 = plaba["1"]

# Get the first PubMed article
article = question_1["15902691"]

# See what information the article contains and decide what to actually chunk (individual sentence/group sentences?)
print("Article type:", type(article))
print("Article keys:", list(article.keys()))

Article type: <class 'dict'>
Article keys: ['Title', 'abstract', 'adaptations']


In [ ]:
# Get the abstract, adaptations, and structure from the article and figure out how PLABA stores OG and simplified text
print("Abstract type:", type(article["abstract"]))
print("Adaptations type:", type(article["adaptations"]))

print("\nABSTRACT PREVIEW:")
print(str(article["abstract"])[:300])

print("\nADAPTATIONS PREVIEW:")
print(str(article["adaptations"])[:300])

Abstract type: <class 'dict'>
Adaptations type: <class 'dict'>

ABSTRACT PREVIEW:
{'1': 'Muscle cramps are a common problem characterized by a sudden, painful, involuntary contraction of muscle.', '2': 'These true cramps, which originate from peripheral nerves, may be distinguished from other muscle pain or spasm.', '3': 'Medical history, physical examination, and a limited labor

ADAPTATIONS PREVIEW:
{'adaptation2': {'1': 'Muscle cramps are a common problem represented by sudden, painful, involuntary muscle contractions.', '2': 'These true cramps, coming from nerves outside the brain and spinal cord, may be identifiable from other muscle pains.', '3': 'Medical history, physical check-up, and lab


In [ ]:
# Test individual sentence retrieval from the abstract and adaptations
# Access the original medical sentence
original = article["abstract"]["2"]

# Access its simplified adaptation
simplified = article["adaptations"]["adaptation2"]["2"]

print("ORIGINAL:")
print(original)

print("\nSIMPLIFIED:")
print(simplified)

ORIGINAL:
These true cramps, which originate from peripheral nerves, may be distinguished from other muscle pain or spasm.

SIMPLIFIED:
These true cramps, coming from nerves outside the brain and spinal cord, may be identifiable from other muscle pains.


In [ ]:
# Tells us how many OG sentences exist and how many adaptations exist for each OG sentence
print("Adaptation versions:")
print(list(article["adaptations"].keys()))

print("\nOriginal sentences:", len(article["abstract"]))

for version, sentences in article["adaptations"].items():
    print(version, ":", len(sentences), "entries")

Adaptation versions:
['adaptation2']

Original sentences: 7
adaptation2 : 7 entries


In [9]:
# Match original sentences with their simplified versions
sentence_pairs = []

for sentence_id, original in article["abstract"].items():

    if sentence_id in article["adaptations"]["adaptation2"]:

        simplified = article["adaptations"]["adaptation2"][sentence_id]

        sentence_pairs.append({
            "sentence_id": sentence_id,
            "original": original,
            "simplified": simplified
        })

print("Total sentence pairs:", len(sentence_pairs))
print(sentence_pairs[1])

Total sentence pairs: 7
{'sentence_id': '2', 'original': 'These true cramps, which originate from peripheral nerves, may be distinguished from other muscle pain or spasm.', 'simplified': 'These true cramps, coming from nerves outside the brain and spinal cord, may be identifiable from other muscle pains.'}


In [ ]:
# Sends 7 paired sentences to Ollama and returns information into JSON format
import json
from ollama import chat

model_name = "qwen3"

# Ask Ollama to identify medical terminology.
prompt = f"""
Compare these original biomedical sentences
with their simplified versions.

Identify up to 8 difficult medical terms
that appear in the ORIGINAL text.

For each term, provide:
- term: the exact medical term
- definition: a short plain-language explanation
- sentence_id: the original sentence number

Use the simplified text as guidance.
Do not invent terms that aren't in the original.
Return only a JSON object with a "terms" list.

Sentence pairs:
{json.dumps(sentence_pairs)}
"""

response = chat(
    model=model_name,
    messages=[
        {"role": "user", "content": prompt}
    ],
    format="json",
    options={"temperature": 0}
)

extracted = json.loads(response["message"]["content"])

for item in extracted["terms"]:
    print(item["term"], ":", item["definition"])


peripheral nerves : nerves outside the brain and spinal cord
limited laboratory screen : a basic set of laboratory tests
benign : not harmful or cancerous
side-effect profile : the range and severity of side effects a drug may cause
membrane-stabilizing drugs : drugs that help prevent abnormal electrical activity in cells
therapeutic trials : tests or experiments to evaluate the effectiveness of treatments
pathophysiology : the study of the functional changes that occur in a disease or condition
quinine sulfate : a medication used to treat malaria and muscle cramps


In [ ]:
# Collect 50 medical terms from different question groups by looping through PLABA to cover wider range vocabulary
import json
from ollama import chat

glossary = []
seen_terms = set()

# Examine articles from different medical question groups.
for question_id, question_data in list(plaba.items())[:20]:

    # Get the first article in this question group.
    article_id = next(
        (key for key in question_data
         if key not in ["question", "question_type"]),
        None
    )

    if article_id is None:
        continue

    article_data = question_data[article_id]

    # Get the first available adaptation version.
    versions = article_data["adaptations"]
    if not versions:
        continue

    version = next(iter(versions))
    original_sentences = article_data["abstract"]
    simplified_sentences = versions[version]

    # Match original sentences to their adaptations.
    pairs = []

    for sentence_id, original in original_sentences.items():
        if sentence_id in simplified_sentences:
            pairs.append({
                "original": original,
                "simplified": simplified_sentences[sentence_id]
            })

    if not pairs:
        continue

    prompt = f"""
    Extract up to 8 difficult medical terms from
    these original biomedical sentences.

    Use the simplified sentences to help explain
    their meanings in everyday language.

    Return a JSON object with a "terms" list.
    Each item must have:
    - "term"
    - "definition"

    Only select terms appearing in the original
    sentences. Do not invent medical information.

    {json.dumps(pairs)}
    """

    try:
        response = chat(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            format="json",
            options={"temperature": 0}
        )

        data = json.loads(response["message"]["content"])

        original_text = " ".join(
            str(s) for s in original_sentences.values()
        ).lower()

        for item in data.get("terms", []):
            term = item.get("term", "").strip()
            definition = item.get("definition", "").strip()

            if not term or not definition:
                continue

            # Only keep terms actually found in the article.
            if term.lower() not in original_text:
                continue

            # Avoid duplicate terms.
            if term.lower() in seen_terms:
                continue

            seen_terms.add(term.lower())

            glossary.append({
                "term": term,
                "definition": definition,
                "question_id": question_id,
                "pmid": article_id,
                "review_status": "unverified"
            })

    except Exception as error:
        print("Error processing article:", article_id, error)

    print("Vocabulary collected:", len(glossary))

    if len(glossary) >= 50:
        break

# Keep at most 50 entries.
glossary = glossary[:50]

print("\nTotal vocabulary terms:", len(glossary))

Vocabulary collected: 7
Vocabulary collected: 15
Vocabulary collected: 23
Vocabulary collected: 31
Vocabulary collected: 39
Vocabulary collected: 47
Vocabulary collected: 55

Total vocabulary terms: 50


In [ ]:
import json
from pathlib import Path

# Locate the project's data folder
project_root = Path.cwd()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

# Path to your new JSON file
output_path = project_root / "data" / "m2" / "m2-glossary.json"

# Save and write all 50 vocabulary entries into existing JSON file
with open(output_path, "w", encoding="utf-8") as file:
    json.dump(glossary, file, indent=2, ensure_ascii=False)

print("Saved", len(glossary), "terms to", output_path)

Saved 50 terms to /Users/andrewnguyen/fa26-ai-team-3/data/m2/m2-glossary.json


In [15]:
# Test two embedding models from the Sentence Transformers library to see if they produce different vector representations for the same text
from sentence_transformers import SentenceTransformer

# Load the first embedding model
model1 = SentenceTransformer("all-MiniLM-L6-v2")

# Load the second embedding model
model2 = SentenceTransformer("all-mpnet-base-v2")

# Check embedding dimensions
print("MiniLM:", model1.get_sentence_embedding_dimension())
print("MPNet:", model2.get_sentence_embedding_dimension())

MiniLM: 384
MPNet: 768


In [16]:
# Store glossary using both embedding models in separate ChromaDB collections for comparison
import json
import chromadb
from pathlib import Path

# Find the project folder
project_root = Path.cwd()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

# Load our 50 medical terms
with open(project_root / "data/m2/m2-glossary.json", "r") as f:
    glossary = json.load(f)

# Prepare the same vocabulary for both models
documents = [
    item["term"] + ": " + item["definition"]
    for item in glossary
]

ids = [f"term_{i}" for i in range(len(glossary))]

# Generate embeddings using each model
embeddings1 = model1.encode(documents).tolist()
embeddings2 = model2.encode(documents).tolist()

# Create separate ChromaDB collections
compare_client = chromadb.PersistentClient(
    path=str(project_root / "data/m2/embedding_compare_db")
)

collection1 = compare_client.get_or_create_collection(
    name="minilm_comparison",
    embedding_function=None
)

collection2 = compare_client.get_or_create_collection(
    name="mpnet_comparison",
    embedding_function=None
)

# Store the same glossary with different embeddings
collection1.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings1
)

collection2.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings2
)

print("MiniLM entries:", collection1.count())
print("MPNet entries:", collection2.count())

MiniLM entries: 50
MPNet entries: 50


In [24]:
# Test queries to see if the two models retrieve the same medical terms
import pandas as pd

# Queries and their expected medical terms
test_queries = [
    (
        "Nerves that connect the body to the central nervous system",
        "Peripheral nerves"
    ),
    (
        "My muscle suddenly tightens without me controlling it",
        "Spasm"
    ),
    (
        "A doctor orders a few basic lab tests to check my health",
        "Limited laboratory screen"
    ),
    (
        "A muscle contracts suddenly and unexpectedly",
        "Spasm"
    ),
    (
        "How do I repair my phone?",
        None
    )
]

# Match stored IDs to medical terms
id_to_term = {
    f"term_{i}": item["term"]
    for i, item in enumerate(glossary)
}

# Search a collection using its embedding model
def search_model(model, collection, query):
    query_embedding = model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=1
    )

    result_id = results["ids"][0][0]
    return id_to_term[result_id]

# Store our comparison results
comparison = []

for query, expected in test_queries:

    minilm_result = search_model(model1, collection1, query)
    mpnet_result = search_model(model2, collection2, query)

    # We cannot score unrelated queries with this
    # nearest-neighbor search alone.
    if expected is None:
        minilm_correct = None
        mpnet_correct = None
    else:
        minilm_correct = minilm_result.lower() == expected.lower()
        mpnet_correct = mpnet_result.lower() == expected.lower()

    comparison.append({
        "Query": query,
        "Expected": expected,
        "MiniLM Result": minilm_result,
        "MiniLM Correct": minilm_correct,
        "MPNet Result": mpnet_result,
        "MPNet Correct": mpnet_correct
    })

results_df = pd.DataFrame(comparison)
display(results_df)


,Query,Expected,MiniLM Result,MiniLM Correct,MPNet Result,MPNet Correct
0,Nerves that connect the body to the central ne...,Peripheral nerves,Peripheral nerves,True,Peripheral nerves,True
1,My muscle suddenly tightens without me control...,Spasm,Spasm,True,Spasm,True
2,A doctor orders a few basic lab tests to check...,Limited laboratory screen,Limited laboratory screen,True,Limited laboratory screen,True
3,A muscle contracts suddenly and unexpectedly,Spasm,Spasm,True,Spasm,True
4,How do I repair my phone?,None,complications,None,Arthroscopic treatment,None
